Instalacija biblioteke cornac

In [1]:
!pip install cornac

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.6/29.6 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.0/192.0 kB 13.6 MB/s eta 0:00:00


Korištenjem biblioteka zipfile i urllib.request se dohvaća baza podataka MovieLens1m. Nakon toga se njezini sadržaji obrađuju korištenjem biblioteke pandas. Varijable genres, ages i occupations sadrže rječnike za mapiranje značajki u numeričke formate prema uputama korištenja baze podataka.

In [2]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import urllib.request
import zipfile
import os
ml_1m_url = "https://github.com/matej-glavas/NCF-movie-rec/raw/refs/heads/main/ml-1m.zip"
urllib.request.urlretrieve(ml_1m_url, "ml-1m.zip")
with zipfile.ZipFile("ml-1m.zip", "r") as ml_1m_zip:
  ml_1m_zip.extractall()
movies = pd.read_csv("ml-1m/movies.dat", sep="::", names=["movie_id", "title", "genres"], engine="python", encoding="latin-1")
users = pd.read_csv("ml-1m/users.dat", sep="::", names=["user_id", "gender", "age", "occupation", "zipcode"], engine="python", encoding="latin-1")
ratings = pd.read_csv("ml-1m/ratings.dat", sep="::", names=["user_id", "movie_id", "rating", "timestamp"], engine="python", encoding="latin-1")
genres = {"Action": 0, "Adventure": 1, "Animation": 2, "Children's": 3, "Comedy": 4, "Crime": 5, "Documentary": 6, "Drama": 7, "Fantasy": 8, "Film-Noir": 9, "Horror": 10, "Musical": 11, "Mystery": 12, "Romance": 13, "Sci-Fi": 14, "Thriller": 15, "War": 16, "Western": 17}
ages = {1: 0 , 18:  1, 25:  2, 35:  3 , 45:  4 , 50:  5, 56:  6};
occupations = {0:  "other or not specified",  1:  "academic/educator",  2:  "artist",  3:  "clerical/admin",  4:  "college/grad student",  5:  "customer service",  6:  "doctor/health care",  7:  "executive/managerial",  8:  "farmer",  9:  "homemaker", 10:  "K-12 student", 11:  "lawyer" , 12:  "programmer", 13:  "retired", 14:  "sales/marketing", 15:  "scientist", 16:  "self-employed", 17:  "technician/engineer", 18:  "tradesman/craftsman" , 19:  "unemployed", 20:  "writer"}

Kreacija tenzora iz objekta tipa pd.DataFrame kako bi se podatci mogli koristiti u biblioteci PyTorch. Žanrovi za film se obrađuju u listi duljine broja žanrova u rječniku žanrova, gdje 0 predstavlja da žanr nije, a 1 da žanr je prisutan u filmu. Iz baze podataka se ne koriste podaci o vremenu ocjenjivanja i mjestu stanovanja korisnika.

In [3]:
def create_movies_tensor(movies, genres):
  movies_values = movies.values
  for i in range(len(movies_values)):
    movies_values[i, 2] = movies_values[i, 2].split("|")
    ind_genres = []
    for j in range(len(genres)):
      ind_genres.append(0)
    for j in range(len(movies_values[i, 2])):
      ind_genres[genres[movies_values[i,2][j]]] = 1
    movies_values[i, 2] = ind_genres
  movies_values = np.delete(movies_values, 1, axis=1)
  movie_ids = movies_values[:, 0].astype(np.float32)
  movie_ids_tensor = torch.tensor(movie_ids).unsqueeze(1)
  movie_genres = np.array(list(movies_values[:, 1]), dtype=np.float32)
  movie_genres_tensor = torch.tensor(movie_genres)
  movies_tensor = torch.cat((movie_ids_tensor, movie_genres_tensor), 1)
  return movies_tensor
def create_users_tensor(users):
  user_values = users.values
  for i in range(len(user_values)):
    if user_values[i, 1] == "M":
      user_values[i, 1] = 0
    else:
      user_values[i, 1] = 1
    user_values[i, 2] = ages[user_values[i, 2]]
  user_values = np.delete(user_values, 4, axis=1)
  user_list = user_values.astype(np.float32)
  user_tensor = torch.tensor(user_list)
  return user_tensor
def create_ratings_tensor(ratings):
  rating_values = ratings.values
  rating_values = np.delete(rating_values, 3, axis=1)
  rating_list = rating_values.astype(np.float32)
  rating_tensor = torch.tensor(rating_list)
  return rating_tensor

movies_tensor = create_movies_tensor(movies, genres)
user_tensor = create_users_tensor(users)
rating_tensor = create_ratings_tensor(ratings)


Kreacija klase Model. Klasa sadrži dvije varijable o latentnim reprezentacijama, jedna o filmovima i jedna o korisnicima. Sadrži tri varijable tipa Linear koje predstavljaju povezane slojeve neuronskih mreža koje kombiniraju ulazne podatke (smanjuje se dimenzija sa 149 na 128, pa 64 i na kraju 1). Sadrži dvije varijable tipa BatchNorm1d koje normaliziraju podatke radi bržeg i stabilnijeg učenja te varijablu tipa Dropout kako bi se ponekad odbacili pojedini neuroni kako bi ostali znali raditi u drugačijim uvjetima umjesto učenja podataka na pamet.

U metodi forward se u vector povezuju latentne reprezentacije te podaci o korisnicima i filmovima, ako se ne zada user_id onda kreće od nul-vektora. Zatim taj vektor prolazi kroz linearne slojeve, normalizaciju, aktivacijsku funkciju i dropout regularizaciju kako bi generirao konačnu predikciju ocjene.

Varijabla criterion sadrži funkciju gubitka (srednja kvadratna pogreška) koja mjeri odstupanje predviđenih od stvarnih ocjena. Varijabla optimizer inicijalizira Adam optimizacijski algoritam koji prilagođuje težine tijekom učenja kako bi se minimizirao gubitak.

Varijable user_id_per_rating i movie_id_per_rating sadrže podatke o identifikatorima korisnika i filmova na svim retcima u tenzoru o ocjenama.

Pomoću funkcije randperm podaci se nasumično dijele u skup za treniranje(80%), validaciju(10%) i testiranje(10%). Budući da isti korisnik ocjenjuje više filmova, dio njegovih ocjena može završiti u skupu za učenje, a dio u testnom skupu. To znači da model pri testiranju već „poznaje” korisnika i dio njegova ponašanja, što se razlikuje od hladnog starta.

Varijabla scheduler smanjuje stopu učenja za polovinu svakih 20 epoha kako bi preciznije podešavao težine pred kraj treniranja.

In [4]:
class Model(nn.Module):
  def __init__(self, movies, users):
    super().__init__()
    self.max_movies_id = int(movies.values[:, 0].max())
    self.max_users_id = int(users.values[:, 0].max())
    self.movies_embedding = nn.Embedding(self.max_movies_id + 1, 64)
    self.users_embedding = nn.Embedding(self.max_users_id + 1, 64)
    self.linear1 = nn.Linear(149, 128)
    self.linear2 = nn.Linear(128, 64)
    self.linear3 = nn.Linear(64, 1)
    self.batchnorm1 = nn.BatchNorm1d(128)
    self.batchnorm2 = nn.BatchNorm1d(64)
    self.dropout = nn.Dropout(0.4)
  def forward(self, user_id, movie_id, user_details, movie_details):
    if user_id is not None:
      users_embedding = self.users_embedding(user_id.long())
    else:
      users_embedding = torch.zeros(1, 64)
    vector = torch.cat([self.movies_embedding(movie_id.long()), users_embedding, user_details, movie_details], dim=1)
    vector = self.dropout(torch.relu(self.batchnorm1(self.linear1(vector))))
    vector = self.dropout(torch.relu(self.batchnorm2(self.linear2(vector))))
    vector = self.linear3(vector)
    vector = torch.sigmoid(vector) * 5 + 0.5
    return vector

model = Model(movies, users)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

user_dict = {}
for i in range(len(user_tensor)):
    user_id = int(user_tensor[i, 0].item())
    user_dict[user_id] = i
movie_dict = {}
for i in range(len(movies_tensor)):
    movie_id = int(movies_tensor[i, 0].item())
    movie_dict[movie_id] = i

user_id_per_rating = []
movie_id_per_rating = []
for i in range(len(rating_tensor)):
    user_id  = int(rating_tensor[i, 0])
    movie_id = int(rating_tensor[i, 1])
    user_id_per_rating.append(user_dict[user_id])
    movie_id_per_rating.append(movie_dict[movie_id])
user_id_per_rating  = torch.tensor(user_id_per_rating,  dtype=torch.long)
movie_id_per_rating = torch.tensor(movie_id_per_rating, dtype=torch.long)

torch.manual_seed(100)
np.random.seed(100)

train_data_len = int(0.8*len(rating_tensor))
val_data_len = int(0.1*len(rating_tensor))
perm = torch.randperm(len(rating_tensor))
train_data = perm[:train_data_len]
val_data = perm[train_data_len:train_data_len+val_data_len]
test_data = perm[train_data_len + val_data_len:]

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

Pomoću klase TensorDataset se podaci korišteni za treniranje spremaju u jedan objekt, a DataLoader dijeli te podatke u mini grupe radi bolje optimizacije parametara mreže.

Prvi korak učenja je resetiranje gradijenta. Zatim se računa funkcija gubitka i vraća se sloj po sloj do početka mreže(backward propagation). Na kraju se provodi korekcija svih težina. Učenje se provodi na 60 epoha, svakih 5 provodi se validacija. Validacija se provodi na drugačijem skupu podataka kako bi se izbjegla prenaučenost. Nakon učenja provodi se evaluacija na testnom skupu podataka.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_tensordataset = TensorDataset(rating_tensor[train_data, 0], rating_tensor[train_data, 1], user_id_per_rating[train_data], movie_id_per_rating[train_data], rating_tensor[train_data, 2])
train_dataloader = DataLoader(train_tensordataset, batch_size=2048, shuffle=True)
epochs = 25
for epoch in range(epochs):
  model.train()
  total_loss = 0
  for user_id, movie_id, user_index, movie_index, rating in train_dataloader:
    optimizer.zero_grad()
    user_details  = user_tensor[user_index, 1:]
    movie_details = movies_tensor[movie_index, 1:]
    predictions = model(
      user_id,
      movie_id,
      user_details,
      movie_details
    )
    loss = criterion(predictions.squeeze(), rating.float())
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
  if (epoch + 1)% 5 == 0 and epoch != 0:
    model.eval()
    with torch.no_grad():
      user_details_val  = user_tensor[user_id_per_rating[val_data], 1:]
      movie_details_val = movies_tensor[movie_id_per_rating[val_data], 1:]
      predictions_val = model(
        rating_tensor[val_data, 0],
        rating_tensor[val_data, 1],
        user_details_val,
        movie_details_val
      )
      loss_val = criterion(predictions_val.squeeze(), rating_tensor[val_data, 2].float())
      loss_val = torch.sqrt(loss_val)
      rmse_train = torch.sqrt(torch.tensor(total_loss/len(train_dataloader)))
    print(f"Epoch {epoch+1}/{epochs}, Train RMSE: {rmse_train.item():.4f} | Val RMSE: {loss_val.item():.4f}")
  else:
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_dataloader):.4f}")
  scheduler.step()


model.eval()
with torch.no_grad():
  user_details_test  = user_tensor[user_id_per_rating[test_data], 1:]
  movie_details_test = movies_tensor[movie_id_per_rating[test_data], 1:]
  predictions_test = model(
    rating_tensor[test_data, 0],
    rating_tensor[test_data, 1],
    user_details_test,
    movie_details_test
  )
  loss_test = criterion(predictions_test.squeeze(), rating_tensor[test_data, 2].float())
  loss_test = torch.sqrt(loss_test)
  print(f"Test RMSE: {loss_test.item():.4f}")

Epoch 1/25, Loss: 1.2429
Epoch 2/25, Loss: 1.0605
Epoch 3/25, Loss: 0.9551
Epoch 4/25, Loss: 0.8879
Epoch 5/25, Train RMSE: 0.9238 | Val RMSE: 0.9124
Epoch 6/25, Loss: 0.8351
Epoch 7/25, Loss: 0.8217
Epoch 8/25, Loss: 0.8110
Epoch 9/25, Loss: 0.7998
Epoch 10/25, Train RMSE: 0.8857 | Val RMSE: 0.8814
Epoch 11/25, Loss: 0.7692
Epoch 12/25, Loss: 0.7568
Epoch 13/25, Loss: 0.7461
Epoch 14/25, Loss: 0.7367
Epoch 15/25, Train RMSE: 0.8540 | Val RMSE: 0.8645
Epoch 16/25, Loss: 0.7229
Epoch 17/25, Loss: 0.7159
Epoch 18/25, Loss: 0.7101
Epoch 19/25, Loss: 0.7043
Epoch 20/25, Train RMSE: 0.8362 | Val RMSE: 0.8589
Epoch 21/25, Loss: 0.6715
Epoch 22/25, Loss: 0.6623
Epoch 23/25, Loss: 0.6574
Epoch 24/25, Loss: 0.6520
Epoch 25/25, Train RMSE: 0.8047 | Val RMSE: 0.8574
Test RMSE: 0.8599


Spremanje modela

In [ ]:
model_name = "model_v1.pth"
torch.save(model.state_dict(), model_name)
print(f"Model je uspješno spremljen na: {model_name}")

Model je uspješno spremljen na: model_v1.pth


Učitavanje istreniranog modela

In [5]:
model_url = "https://github.com/matej-glavas/NCF-movie-rec/raw/refs/heads/main/model_v1.pth"
model_path = "model_v1.pth"
if not os.path.exists(model_path):
  urllib.request.urlretrieve(model_url, model_path)
model.load_state_dict(torch.load(model_path, map_location="cpu"))
model.eval()
print(f"Model je uspješno učitan sa: {model_path}")

Model je uspješno učitan sa: model_v1.pth


Korisnički unos demografskih podataka i žanrova filmova. Podaci se zatim pretvaraju u tenzore i provodi se predikcija ocjena za filmove koji sadrže barem jedan od navedenih žanrova. Filmovi se sortiraju prema ocjenama i ispisuje se pet najbolje ocijenjenih.

In [6]:
print("Gender? (M/F)" )
user_gender = input()
while(user_gender.capitalize() != "M" and user_gender.capitalize() != "F"):
  print("Gender? (M/F)" )
  user_gender = input()
if user_gender.capitalize() == "M":
  user_gender = 0
else:
  user_gender = 1
print("Age?\n  1: Under 18\n 18:  18-24\n 25:  25-34\n 35:  35-44\n 45:  45-49\n 50:  50-55\n 56:  56+")
user_age = input()
allowed_ages = ["1", "18", "25", "35", "45", "50", "56"]
while user_age not in allowed_ages:
  print("Age? ")
  user_age = input()
print("Occupation?\n  0:  other\n  1:  academic/educator\n  2:  artist\n  3:  clerical/admin\n  4:  college/grad student\n  5:  customer service\n  6:  doctor/health care\n  7:  executive/managerial\n  8:  farmer\n  9:  homemaker\n 10:  K-12 student\n 11:  lawyer\n 12:  programmer\n 13:  retired\n 14:  sales/marketing\n 15:  scientist\n 16:  self-employed\n 17:  technician/engineer\n 18:  tradesman/craftsman\n 19:  unemployed\n 20:  writer\n")
user_occupation = input()
allowed_occupations = []
for i in range(21):
  allowed_occupations.append(str(i))
while user_occupation not in allowed_occupations:
  print("Occupation? ")
  user_occupation = input()
details = np.array([int(user_gender), ages[int(user_age)], int(user_occupation)]).astype(np.float32)
current_user_details = torch.tensor(details).unsqueeze(0)

print("Genres?\n  Action: 0\n  Adventure: 1\n  Animation: 2\n  Children's: 3\n  Comedy: 4\n  Crime: 5\n  Documentary: 6\n  Drama: 7\n  Fantasy: 8\n  Film-Noir: 9\n  Horror: 10\n  Musical: 11\n  Mystery: 12\n  Romance: 13\n  Sci-Fi: 14\n  Thriller: 15\n  War: 16\n  Western: 17")
genre = input()
allowed_genres = []
user_genres = set()
for i in range(18):
  allowed_genres.append(str(i))
if genre in allowed_genres:
  user_genres.add(int(genre))
while(len(user_genres) == 0):
  print("Genre? ")
  genre = input()
  if genre in allowed_genres:
    user_genres.add(int(genre))
while genre != "X":
  print("Genre? Type X to stop.")
  genre = input().upper()
  if genre in allowed_genres:
    user_genres.add(int(genre))
details = []
for i in range(18):
  if i in user_genres:
    details.append(1)
  else:
    details.append(0)
details = np.array(details).astype(np.float32)
current_movie_details = torch.tensor(details).unsqueeze(0)
recommended_movies = []
model.eval()
with torch.no_grad():
  for i in range(len(movies_tensor)):
    movie_details = movies_tensor[i, 1:].unsqueeze(0)
    movie_id = movies_tensor[i, 0].unsqueeze(0).long()
    if torch.logical_and(current_movie_details, movie_details).sum() >= 1:
      predictions = model(None, movie_id, current_user_details, movie_details)
      recommended_movies.append((movie_id.item(), predictions.item()))
recommended_movies.sort(key=lambda x: x[1], reverse=True)
print("Recommended movies:")
for i in range(5):
  for j in range(len(movies)):
    if str(recommended_movies[i][0]) == str(movies.values[j, 0]):
      print(movies.values[j][1])
      continue

Gender? (M/F)
m
Age?
  1: Under 18
 18:  18-24
 25:  25-34
 35:  35-44
 45:  45-49
 50:  50-55
 56:  56+
18
Occupation?
  0:  other
  1:  academic/educator
  2:  artist
  3:  clerical/admin
  4:  college/grad student
  5:  customer service
  6:  doctor/health care
  7:  executive/managerial
  8:  farmer
  9:  homemaker
 10:  K-12 student
 11:  lawyer
 12:  programmer
 13:  retired
 14:  sales/marketing
 15:  scientist
 16:  self-employed
 17:  technician/engineer
 18:  tradesman/craftsman
 19:  unemployed
 20:  writer

4
Genres?
  Action: 0
  Adventure: 1
  Animation: 2
  Children's: 3
  Comedy: 4
  Crime: 5
  Documentary: 6
  Drama: 7
  Fantasy: 8
  Film-Noir: 9
  Horror: 10
  Musical: 11
  Mystery: 12
  Romance: 13
  Sci-Fi: 14
  Thriller: 15
  War: 16
  Western: 17
15
Genre? Type X to stop.
x
Recommended movies:
Usual Suspects, The (1995)
Matrix, The (1999)
Close Shave, A (1995)
Silence of the Lambs, The (1991)
Reservoir Dogs (1992)


Usporedba istreniranog modela s klasičnim suradničkim filtriranjem koristeći metrike RMSE i MAE. Za klasično suradničko filtriranje koristi se biblioteka cornac. Podatci za treniranje izvlače se iz tenzora ocjena u obliku liste, zatim se pretvaraju u string format koji se koristi za kreaciju dataseta i treniranje SVD modela s 50 latentnih faktora. Evaluacija se provodi ručno prolazkom kroz parove unutar testnog skupa podataka. RMSE jače penalizira veće pogreške od MAE jer kvadrira odstupanja, dok MAE tretira sva odstupanja jednako.

In [7]:
import cornac
import io, sys
from cornac.eval_methods import RatioSplit
from cornac.models import PMF
from cornac.metrics import RMSE, MAE

train_data_cornac = list(zip(
    rating_tensor[train_data, 0].int().tolist(),
    rating_tensor[train_data, 1].int().tolist(),
    rating_tensor[train_data, 2].tolist()
))
train_data_cornac_list = []
for user, movie, rating in train_data_cornac:
    train_data_cornac_list.append((str(int(user)), str(int(movie)), rating))

cornac_reader = cornac.data.Reader()
cornac_dataset = cornac.data.Dataset.from_uir(train_data_cornac_list)
pmf = PMF(k=50, seed=100, verbose=False)
pmf.fit(cornac_dataset)


test_users  = rating_tensor[test_data, 0].int().tolist()
test_movies = rating_tensor[test_data, 1].int().tolist()
test_real   = rating_tensor[test_data, 2].tolist()

preds, reals = [], []
for user, movie, rating in zip(test_users, test_movies, test_real):
    user_raw = str(int(user))
    movie_raw = str(int(movie))
    if user_raw in cornac_dataset.uid_map and movie_raw in cornac_dataset.iid_map:
        u_idx = cornac_dataset.uid_map[user_raw]
        m_idx = cornac_dataset.iid_map[movie_raw]
        try:
            pred = pmf.score(u_idx, m_idx)
            preds.append(pred)
            reals.append(rating)
        except Exception as e:
            pass

preds = np.array(preds)
reals = np.array(reals)
pmf_rmse = np.sqrt(((preds - reals) ** 2).mean())
pmf_mae  = np.abs(preds - reals).mean()
print(f"PMF RMSE: {pmf_rmse:.4f}")
print(f"PMF MAE:  {pmf_mae:.4f}")

print("---------------")

model.eval()
with torch.no_grad():
    user_details_test  = user_tensor[user_id_per_rating[test_data], 1:]
    movie_details_test = movies_tensor[movie_id_per_rating[test_data], 1:]
    predictions_test = model(
      rating_tensor[test_data, 0],
      rating_tensor[test_data, 1],
      user_details_test,
      movie_details_test
    ).squeeze()
    real_ratings = rating_tensor[test_data, 2]

    rmse_ncf = torch.sqrt(nn.MSELoss()(predictions_test, real_ratings)).item()
    mae_ncf  = nn.L1Loss()(predictions_test, real_ratings).item()

print(f"NCF RMSE: {rmse_ncf:.4f}")
print(f"NCF MAE:  {mae_ncf:.4f}")

PMF RMSE: 0.8992
PMF MAE:  0.7046
---------------
NCF RMSE: 0.8599
NCF MAE:  0.6720
